### Plant State Comparison: Recommended vs Actual Action Contexts

**Objective**: Compare the **actual PV values** (plant state) at test episode action timestamps with the PV values at the matched historical episode action timestamps.

**Key Question**: When the similarity approach recommends an action based on a historical match, was the plant actually in a similar state (similar PV readings) during both episodes?

**3 Visualization Types**:
1. **Radar/Spider Charts** — Overlay actual PV values (normalized to operating range for comparability) for selected tags: test vs matched historical
2. **Grouped Bar Charts** — Side-by-side raw PV values per tag, with operating limit reference lines
3. **Heatmap of PV Differences** — Matrix view: rows = action timestamps, columns = tags, color = PV difference as % of operating range

**Configurable**: Tag list can be changed in the configuration cell below.

In [15]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# ==============================================================================
# CONFIGURATION — Change these to control which tags appear on the plots
# ==============================================================================

# Tags to compare in plant state visualizations (add/remove as needed)
COMPARISON_TAGS = [
    '03LIC_1071',   # Target tag (Level Controller)
    '03LIC_1016',   # Level Controller (same family)
    '03PIC_1013',   # Pressure Controller (highly operated)
]

# Which similarity approach results to use
RESULTS_DIR = '/home/h604827/ControlActions/RESULTS/similarity_test_results/no_deviation_episodes_v3_merged'

# Data paths
DATA_DIR = '/home/h604827/ControlActions/DATA'
RESULTS_BASE = '/home/h604827/ControlActions/RESULTS'

print(f"Configured comparison tags: {COMPARISON_TAGS}")
print(f"Results directory: {RESULTS_DIR}")

Configured comparison tags: ['03LIC_1071', '03LIC_1016', '03PIC_1013']
Results directory: /home/h604827/ControlActions/RESULTS/similarity_test_results/no_deviation_episodes_v3_merged


### Load Data

Load all required data sources:
- **Training context**: Context vectors for all historical operator actions (from similarity approach)
- **Recommendation results**: Per-episode Excel with actual vs recommended actions (with matched_episode_id)
- **PV/OP time series**: For building test episode contexts at runtime
- **Supporting data**: Operating limits, SSD data, events, episodes table

In [16]:
# Load training context (historical action contexts from similarity approach)
training_context_df = pd.read_csv(f'{RESULTS_DIR}/similarity_context_training_merged.csv')
training_context_df['action_timestamp'] = pd.to_datetime(training_context_df['action_timestamp'])
training_context_df['alarm_start'] = pd.to_datetime(training_context_df['alarm_start'])
training_context_df['alarm_end'] = pd.to_datetime(training_context_df['alarm_end'])
training_context_df['deviation_start'] = pd.to_datetime(training_context_df['deviation_start'])
print(f"Training context loaded: {training_context_df.shape}")
print(f"  Episodes: {training_context_df['episode_id'].nunique()}")
print(f"  Actions: {len(training_context_df)}")

# Load recommendation results (per-episode Excel sheets)
results_xls = pd.ExcelFile(f'{RESULTS_DIR}/operator_action_recommendations_by_episode.xlsx')
test_episode_ids = [int(s.replace('episode_', '')) for s in results_xls.sheet_names]
print(f"\nTest episodes with results: {len(test_episode_ids)}")

# Load all episode result sheets into one DataFrame
all_results = []
for sheet_name in results_xls.sheet_names:
    df = pd.read_excel(results_xls, sheet_name=sheet_name)
    all_results.append(df)
all_results_df = pd.concat(all_results, ignore_index=True)
all_results_df['action_time'] = pd.to_datetime(all_results_df['action_time'])
print(f"Total result rows: {len(all_results_df)}")

# Load PV/OP time series data
pv_op_data_df = pd.read_parquet(f'{DATA_DIR}/03LIC_1071_JAN_2026_filtered.parquet')
pv_op_data_df.index = pd.to_datetime(pv_op_data_df.index)
pv_op_data_df.sort_index(inplace=True)
print(f"\nPV/OP time series loaded: {pv_op_data_df.shape}")

# Load operating limits
operating_limits_df = pd.read_csv(f'{DATA_DIR}/operating_limits.csv')
op_limits = {}
for _, row in operating_limits_df.iterrows():
    tag_name = row['TAG_NAME']
    tag_base = tag_name.replace('.PV', '').replace('.OP', '')
    lower = row['LOWER_LIMIT']
    upper = row['UPPER_LIMIT']
    op_range = upper - lower
    if op_range > 0:
        op_limits[tag_base] = {'lower': lower, 'upper': upper, 'range': op_range}
print(f"Operating limits loaded for {len(op_limits)} tags")

# Load episodes table
episodes_df = pd.read_excel(f'{RESULTS_BASE}/episode_all_operator_action_plots/episodes_all_with_actions_and_deviations.xlsx')
episodes_df['AlarmStart'] = pd.to_datetime(episodes_df['AlarmStart'])
episodes_df['AlarmEnd'] = pd.to_datetime(episodes_df['AlarmEnd'])
print(f"Episodes table loaded: {len(episodes_df)} episodes")

# Load SSD data for deviation start times
ssd_df = pd.read_excel(f'{DATA_DIR}/SSD_1071_SSD_output_1071_7Jan2026.xlsx')
ssd_df['AlarmStart_rounded_minutes'] = pd.to_datetime(ssd_df['AlarmStart_rounded_minutes'])
ssd_df['Tag_First_Transition_Start_minutes'] = pd.to_datetime(ssd_df['Tag_First_Transition_Start_minutes'])
print(f"SSD data loaded: {len(ssd_df)} records")

Training context loaded: (361, 98)
  Episodes: 86
  Actions: 361

Test episodes with results: 33
Total result rows: 846

PV/OP time series loaded: (1718039, 45)
Operating limits loaded for 26 tags
Episodes table loaded: 609 episodes
SSD data loaded: 10107 records


### Helper Functions

Lookup actual PV values from `pv_op_data_df` at both the test episode action timestamps and the matched historical episode action timestamps.

In [17]:
# Constants
ALARM_THRESHOLD = 28.75
TARGET_TAG = '03LIC_1071'
target_upper = op_limits[TARGET_TAG]['upper'] if TARGET_TAG in op_limits else 42.41


def get_pv_at_timestamp(timestamp, pv_tag):
    """Get PV value at or nearest to given timestamp (forward-fill lookup)."""
    try:
        if hasattr(timestamp, 'tzinfo') and timestamp.tzinfo is not None:
            timestamp = timestamp.tz_localize(None)
        if timestamp in pv_op_data_df.index:
            return pv_op_data_df.loc[timestamp, pv_tag]
        idx = pv_op_data_df.index.get_indexer([timestamp], method='ffill')[0]
        if 0 <= idx < len(pv_op_data_df):
            return pv_op_data_df.iloc[idx][pv_tag]
        idx = pv_op_data_df.index.get_indexer([timestamp], method='bfill')[0]
        if 0 <= idx < len(pv_op_data_df):
            return pv_op_data_df.iloc[idx][pv_tag]
        return np.nan
    except Exception:
        return np.nan


def get_pv_values_at_timestamp(timestamp, tags):
    """Get actual PV values for a list of tags at a given timestamp. Returns dict."""
    return {tag: get_pv_at_timestamp(timestamp, f'{tag}.PV') for tag in tags}


def get_matched_historical_timestamp(matched_episode_id, rec_source):
    """
    Get the action_timestamp from the matched historical episode in training data.
    This is the actual timestamp when the historical operator took action.
    """
    candidates = training_context_df[
        (training_context_df['episode_id'] == matched_episode_id) &
        (training_context_df['action_source'] == rec_source)
    ]
    if len(candidates) == 0:
        candidates = training_context_df[training_context_df['episode_id'] == matched_episode_id]
    if len(candidates) == 0:
        return None
    return candidates.iloc[0]['action_timestamp']


print("Helper functions defined.")
print(f"Operating limits available for {len(op_limits)} tags")
print(f"Alarm threshold: {ALARM_THRESHOLD}")

Helper functions defined.
Operating limits available for 26 tags
Alarm threshold: 28.75


### Build Comparison Data for an Episode

For each test episode, at every action timestamp:
1. Get **actual PV values** for the configured tags from `pv_op_data_df` at the test action timestamp
2. Get the **matched historical episode's action timestamp** from training data
3. Get **actual PV values** at that historical timestamp from the same `pv_op_data_df`
4. Compute raw PV differences and % of operating range differences

In [18]:
def build_episode_comparison_data(episode_id, comparison_tags=COMPARISON_TAGS):
    """
    For a given test episode, build PV-based comparison data between the test episode
    and matched historical episodes at each action timestamp.
    
    Returns a list of dicts, one per action timestamp, containing:
    - action metadata (time, source, direction, magnitude, similarity)
    - actual PV values at the test timestamp for each comparison tag
    - actual PV values at the matched historical timestamp for each comparison tag
    - PV differences (raw and as % of operating range)
    """
    ep_results = all_results_df[all_results_df['episode_id'] == episode_id].copy()
    if len(ep_results) == 0:
        print(f"No results found for episode {episode_id}")
        return []
    
    ep_meta = episodes_df[episodes_df['EpisodeID'] == episode_id]
    if len(ep_meta) == 0:
        print(f"No episode metadata found for episode {episode_id}")
        return []
    ep_meta = ep_meta.iloc[0]
    alarm_start = ep_meta['AlarmStart']
    alarm_end = ep_meta['AlarmEnd']
    
    actual_actions = ep_results[ep_results['row_type'] == 'actual'].copy()
    rank1_recs = ep_results[ep_results['row_type'] == 'recommended_rank1'].copy()
    
    comparison_records = []
    
    for _, actual_row in actual_actions.iterrows():
        action_time = actual_row['action_time']
        action_group = actual_row['action_group']
        
        rec_row = rank1_recs[rank1_recs['action_group'] == action_group]
        if len(rec_row) == 0:
            continue
        rec_row = rec_row.iloc[0]
        
        matched_ep_id = rec_row['matched_episode_id']
        rec_source = rec_row['source']
        
        # Get ACTUAL PV values at the test action timestamp
        test_pvs = get_pv_values_at_timestamp(action_time, comparison_tags)
        
        # Get the matched historical action timestamp and its PV values
        hist_timestamp = get_matched_historical_timestamp(matched_ep_id, rec_source)
        if hist_timestamp is not None:
            hist_pvs = get_pv_values_at_timestamp(hist_timestamp, comparison_tags)
        else:
            hist_pvs = {tag: np.nan for tag in comparison_tags}
        
        record = {
            'episode_id': episode_id,
            'action_group': action_group,
            'action_time': action_time,
            'alarm_start': alarm_start,
            'alarm_end': alarm_end,
            'actual_source': actual_row['source'],
            'actual_direction': actual_row['direction'],
            'actual_magnitude': actual_row['magnitude'],
            'actual_type': actual_row['action_type'],
            'rec_source': rec_source,
            'rec_direction': rec_row['direction'],
            'rec_magnitude': rec_row['magnitude'],
            'rec_type': rec_row['action_type'],
            'similarity': rec_row['similarity'],
            'matched_episode_id': matched_ep_id,
            'hist_action_timestamp': hist_timestamp,
            'tag_match': rec_row.get('tag_match', np.nan),
            'direction_match': rec_row.get('direction_match', np.nan),
        }
        
        # Add actual PV values and differences for each comparison tag
        for tag in comparison_tags:
            test_pv = test_pvs[tag]
            hist_pv = hist_pvs[tag]
            limits = op_limits.get(tag, None)
            
            record[f'{tag}_test_pv'] = test_pv
            record[f'{tag}_hist_pv'] = hist_pv
            
            # Raw difference
            if pd.notna(test_pv) and pd.notna(hist_pv):
                record[f'{tag}_pv_diff'] = test_pv - hist_pv
            else:
                record[f'{tag}_pv_diff'] = np.nan
            
            # Difference as % of operating range (normalizes across tags)
            if limits and pd.notna(test_pv) and pd.notna(hist_pv):
                record[f'{tag}_pv_diff_pct'] = (test_pv - hist_pv) / limits['range'] * 100
            else:
                record[f'{tag}_pv_diff_pct'] = np.nan
            
            # Store operating limits for reference
            if limits:
                record[f'{tag}_lower'] = limits['lower']
                record[f'{tag}_upper'] = limits['upper']
                record[f'{tag}_range'] = limits['range']
        
        comparison_records.append(record)
    
    return comparison_records


# Quick test with first available episode
test_ep_id = test_episode_ids[0]
print(f"Testing with episode {test_ep_id}...")
test_comparison = build_episode_comparison_data(test_ep_id)
print(f"  Built {len(test_comparison)} comparison records")
if len(test_comparison) > 0:
    r = test_comparison[0]
    print(f"\n  Action 1 at {r['action_time']}:")
    print(f"    Actual: {r['actual_source']} {'↑' if r['actual_direction']==1 else '↓'} {r['actual_magnitude']:+.1f}")
    print(f"    Recommended: {r['rec_source']} {'↑' if r['rec_direction']==1 else '↓'} {r['rec_magnitude']:+.1f} (sim={r['similarity']:.3f})")
    print(f"    Matched episode: {int(r['matched_episode_id'])}, hist timestamp: {r['hist_action_timestamp']}")
    print(f"\n  PV Values (Test vs Historical):")
    for tag in COMPARISON_TAGS:
        test_pv = r[f'{tag}_test_pv']
        hist_pv = r[f'{tag}_hist_pv']
        diff = r[f'{tag}_pv_diff']
        diff_pct = r[f'{tag}_pv_diff_pct']
        print(f"    {tag}: Test={test_pv:.2f}, Hist={hist_pv:.2f}, Diff={diff:+.2f} ({diff_pct:+.1f}% of range)")

Testing with episode 558...
  Built 19 comparison records

  Action 1 at 2025-05-05 21:02:00:
    Actual: 03LIC_1071 ↑ +2.0
    Recommended: 03LIC_1071 ↑ +4.0 (sim=0.282)
    Matched episode: 468, hist timestamp: 2024-09-21 07:26:00

  PV Values (Test vs Historical):
    03LIC_1071: Test=8.95, Hist=39.92, Diff=-30.97 (-432.1% of range)
    03LIC_1016: Test=54.65, Hist=33.20, Diff=+21.46 (+433.0% of range)
    03PIC_1013: Test=148.67, Hist=162.84, Diff=-14.17 (-18.9% of range)


---
## Plot 1: Radar/Spider Charts — PV Value Fingerprint Comparison

For each action timestamp, overlay the **actual PV values** (normalized to 0-1 operating range) for the configured tags from the test episode vs the matched historical episode. 

- Each axis = one tag's PV value normalized within its operating range (0 = lower limit, 1 = upper limit)
- If the two polygons overlap closely → the plant was in a similar state
- Hover shows the actual raw PV values

In [19]:
def plot_radar_comparison(comparison_records, comparison_tags=COMPARISON_TAGS):
    """
    Radar charts comparing actual PV values (normalized to operating range) for
    test vs historical at each action timestamp.
    """
    n_actions = len(comparison_records)
    if n_actions == 0:
        print("No comparison records to plot.")
        return None
    
    episode_id = comparison_records[0]['episode_id']
    
    cols = min(n_actions, 3)
    rows = (n_actions + cols - 1) // cols
    
    fig = make_subplots(
        rows=rows, cols=cols,
        specs=[[{'type': 'polar'} for _ in range(cols)] for _ in range(rows)],
        subplot_titles=[
            f"Action {r['action_group']}: {r['actual_source']} @ {r['action_time'].strftime('%H:%M')}"
            for r in comparison_records
        ]
    )
    
    tag_labels = [t.replace('03', '').replace('_', '') for t in comparison_tags]
    tag_labels_closed = tag_labels + [tag_labels[0]]
    
    for i, record in enumerate(comparison_records):
        row = i // cols + 1
        col = i % cols + 1
        
        # Normalize PV values to [0, 1] using operating limits
        test_norm = []
        hist_norm = []
        test_raw = []
        hist_raw = []
        for tag in comparison_tags:
            limits = op_limits.get(tag, None)
            t_pv = record.get(f'{tag}_test_pv', np.nan)
            h_pv = record.get(f'{tag}_hist_pv', np.nan)
            
            test_raw.append(t_pv if pd.notna(t_pv) else 0)
            hist_raw.append(h_pv if pd.notna(h_pv) else 0)
            
            if limits and pd.notna(t_pv):
                test_norm.append((t_pv - limits['lower']) / limits['range'])
            else:
                test_norm.append(0)
            if limits and pd.notna(h_pv):
                hist_norm.append((h_pv - limits['lower']) / limits['range'])
            else:
                hist_norm.append(0)
        
        test_norm_closed = test_norm + [test_norm[0]]
        hist_norm_closed = hist_norm + [hist_norm[0]]
        test_raw_closed = test_raw + [test_raw[0]]
        hist_raw_closed = hist_raw + [hist_raw[0]]
        
        fig.add_trace(go.Scatterpolar(
            r=test_norm_closed,
            theta=tag_labels_closed,
            fill='toself',
            name=f'Test (Ep {episode_id})',
            line=dict(color='rgba(31, 119, 180, 0.9)', width=2),
            fillcolor='rgba(31, 119, 180, 0.15)',
            showlegend=(i == 0),
            legendgroup='test',
            customdata=test_raw_closed,
            hovertemplate='%{theta}<br>Norm: %{r:.3f}<br>PV: %{customdata:.2f}<extra>Test Episode</extra>'
        ), row=row, col=col)
        
        fig.add_trace(go.Scatterpolar(
            r=hist_norm_closed,
            theta=tag_labels_closed,
            fill='toself',
            name=f'Historical (Ep {int(record["matched_episode_id"])})',
            line=dict(color='rgba(255, 127, 14, 0.9)', width=2, dash='dash'),
            fillcolor='rgba(255, 127, 14, 0.15)',
            showlegend=(i == 0),
            legendgroup='hist',
            customdata=hist_raw_closed,
            hovertemplate='%{theta}<br>Norm: %{r:.3f}<br>PV: %{customdata:.2f}<extra>Historical Episode</extra>'
        ), row=row, col=col)
    
    fig.update_layout(
        title=f'Episode {episode_id}: PV Value Fingerprint (Normalized to Operating Range)<br>'
              f'<sub>Blue = Test Episode | Orange = Matched Historical | Hover for raw PV values</sub>',
        height=400 * rows,
        showlegend=True,
    )
    
    for i in range(n_actions):
        polar_key = f'polar{i+1}' if i > 0 else 'polar'
        fig.update_layout(**{polar_key: dict(radialaxis=dict(visible=True, range=[0, 1.2]))})
    
    return fig


# Demo
if len(test_comparison) > 0:
    fig_radar = plot_radar_comparison(test_comparison)
    fig_radar.show()

---
## Plot 2: Grouped Bar Charts — Raw PV Values Side by Side

For each action timestamp, show **actual PV values** as grouped bars (test vs historical) for each configured tag. 

Since tags have different scales, each tag gets its own row with consistent y-axis. Operating limits are shown as horizontal dashed lines for reference.

In [20]:
def plot_bar_comparison(comparison_records, comparison_tags=COMPARISON_TAGS):
    """
    Grouped bar chart: actual PV values test vs historical.
    Each tag gets its own row (different y-scales), each action gets a column.
    Operating limits shown as horizontal dashed lines.
    """
    n_actions = len(comparison_records)
    if n_actions == 0:
        print("No comparison records to plot.")
        return None
    
    episode_id = comparison_records[0]['episode_id']
    n_tags = len(comparison_tags)
    max_cols = min(n_actions, 5)
    
    fig = make_subplots(
        rows=n_tags, cols=max_cols,
        subplot_titles=[
            f"Action {comparison_records[i]['action_group']} @ {comparison_records[i]['action_time'].strftime('%H:%M')}"
            if i < max_cols else '' for i in range(max_cols)
        ] + ['' for _ in range((n_tags - 1) * max_cols)],
        vertical_spacing=0.08,
        horizontal_spacing=0.05,
        row_titles=[t.replace('03', '').replace('_', '') for t in comparison_tags],
    )
    
    for i, record in enumerate(comparison_records[:max_cols]):
        col = i + 1
        sim = record.get('similarity', 0)
        
        for j, tag in enumerate(comparison_tags):
            row = j + 1
            t_pv = record.get(f'{tag}_test_pv', 0) or 0
            h_pv = record.get(f'{tag}_hist_pv', 0) or 0
            limits = op_limits.get(tag, None)
            
            fig.add_trace(go.Bar(
                x=['Test', 'Historical'],
                y=[t_pv, h_pv],
                marker_color=['rgba(31, 119, 180, 0.8)', 'rgba(255, 127, 14, 0.8)'],
                showlegend=False,
                text=[f'{t_pv:.2f}', f'{h_pv:.2f}'],
                textposition='outside',
                hovertemplate=f'{tag}<br>%{{x}}: %{{y:.2f}}<extra></extra>'
            ), row=row, col=col)
            
            # Add operating limit lines
            if limits:
                xref = f'x{(row-1)*max_cols + col}' if (row-1)*max_cols + col > 1 else 'x'
                yref = f'y{(row-1)*max_cols + col}' if (row-1)*max_cols + col > 1 else 'y'
                fig.add_hline(
                    y=limits['lower'], line_dash='dot', line_color='red', opacity=0.5,
                    row=row, col=col
                )
                fig.add_hline(
                    y=limits['upper'], line_dash='dot', line_color='red', opacity=0.5,
                    row=row, col=col
                )
                
                # Add alarm threshold for target tag
                if tag == TARGET_TAG:
                    fig.add_hline(
                        y=ALARM_THRESHOLD, line_dash='dash', line_color='red', opacity=0.8,
                        row=row, col=col,
                        annotation_text='Alarm' if col == 1 else None,
                        annotation_position='bottom left'
                    )
    
    fig.update_layout(
        title=f'Episode {episode_id}: Actual PV Values — Test vs Historical<br>'
              f'<sub>Blue = Test Episode | Orange = Matched Historical | Red dashed = Operating Limits / Alarm</sub>',
        height=250 * n_tags,
        width=220 * max_cols + 100,
        showlegend=False,
    )
    
    return fig


# Demo
if len(test_comparison) > 0:
    fig_bar = plot_bar_comparison(test_comparison)
    fig_bar.show()

---
## Plot 3: Heatmap of PV Differences — Bird's Eye View

Matrix view across all action timestamps in an episode:
- **Rows** = action timestamps (chronological)
- **Columns** = configured tags
- **Color** = PV difference as **% of operating range** (test PV - historical PV) / range × 100
  - **White/near-zero** = plant states are very similar for this tag
  - **Blue** = test PV is lower than historical
  - **Red** = test PV is higher than historical

Hover shows raw PV values for both test and historical.

In [21]:
def plot_heatmap_comparison(comparison_records, comparison_tags=COMPARISON_TAGS):
    """
    Heatmap of PV differences (as % of operating range) between test and historical.
    Rows = action timestamps, Columns = tags.
    """
    n_actions = len(comparison_records)
    if n_actions == 0:
        print("No comparison records to plot.")
        return None
    
    episode_id = comparison_records[0]['episode_id']
    
    col_names = [t.replace('03', '').replace('_', '') for t in comparison_tags]
    row_labels = []
    diff_matrix = []
    annotations_text = []
    
    for record in comparison_records:
        time_str = record['action_time'].strftime('%H:%M')
        src = record['actual_source'].replace('03', '').replace('_', '')
        sim = record.get('similarity', 0)
        row_labels.append(f"Act {record['action_group']}: {src} @ {time_str}<br>(sim={sim:.3f})")
        
        row_diffs = []
        row_text = []
        for tag in comparison_tags:
            diff_pct = record.get(f'{tag}_pv_diff_pct', 0) or 0
            row_diffs.append(diff_pct)
            
            t_pv = record.get(f'{tag}_test_pv', np.nan)
            h_pv = record.get(f'{tag}_hist_pv', np.nan)
            raw_diff = record.get(f'{tag}_pv_diff', np.nan)
            row_text.append(
                f"Test PV: {t_pv:.2f}<br>"
                f"Hist PV: {h_pv:.2f}<br>"
                f"Diff: {raw_diff:+.2f}<br>"
                f"% of Range: {diff_pct:+.1f}%"
            )
        
        diff_matrix.append(row_diffs)
        annotations_text.append(row_text)
    
    diff_array = np.array(diff_matrix)
    
    fig = go.Figure(data=go.Heatmap(
        z=diff_array,
        x=col_names,
        y=row_labels,
        colorscale='RdBu_r',
        zmid=0,
        zmin=-30,
        zmax=30,
        text=annotations_text,
        hovertemplate='%{y}<br>%{x}<br>%{text}<extra></extra>',
        colorbar=dict(title='PV Diff<br>(% of range)')
    ))
    
    # Annotate cells with % values
    for i in range(len(diff_matrix)):
        for j in range(len(diff_matrix[i])):
            val = diff_matrix[i][j]
            fig.add_annotation(
                x=col_names[j], y=row_labels[i],
                text=f"{val:+.1f}%",
                showarrow=False,
                font=dict(size=11, color='black' if abs(val) < 15 else 'white')
            )
    
    fig.update_layout(
        title=f'Episode {episode_id}: PV Difference Heatmap (% of Operating Range)<br>'
              f'<sub>Red = test PV higher than historical | Blue = test PV lower | White = similar</sub>',
        height=max(300, 80 * n_actions + 150),
        width=max(500, 180 * len(col_names) + 200),
        xaxis_title='Tags',
        yaxis_title='Action Timestamps',
        yaxis=dict(autorange='reversed'),
    )
    
    return fig


# Demo
if len(test_comparison) > 0:
    fig_heatmap = plot_heatmap_comparison(test_comparison)
    fig_heatmap.show()

---
## Combined Episode Dashboard

Generate all 3 plots for a single episode plus a summary table showing raw PV values and differences.

In [22]:
def generate_episode_dashboard(episode_id, comparison_tags=COMPARISON_TAGS, show=True):
    """
    Generate all 3 PV comparison plots for a single episode.
    Returns the comparison records and figure objects.
    """
    print(f"\n{'='*80}")
    print(f"  Episode {episode_id}: Plant State (PV) Comparison Dashboard")
    print(f"{'='*80}")
    
    records = build_episode_comparison_data(episode_id, comparison_tags)
    
    if len(records) == 0:
        print(f"  No comparison data available for episode {episode_id}.")
        return records, None, None, None
    
    print(f"  {len(records)} action timestamps to compare\n")
    
    # Summary table with raw PV values
    for r in records:
        print(f"  Action {r['action_group']} @ {r['action_time'].strftime('%H:%M')} | "
              f"Actual: {r['actual_source'][-4:]} {'↑' if r['actual_direction']==1 else '↓'}{abs(r['actual_magnitude']):4.1f} | "
              f"Rec: {r['rec_source'][-4:]} {'↑' if r['rec_direction']==1 else '↓'}{abs(r['rec_magnitude']):4.1f} | "
              f"Sim: {r['similarity']:.3f} | "
              f"Matched Ep: {int(r['matched_episode_id'])}")
        for tag in comparison_tags:
            t_pv = r.get(f'{tag}_test_pv', np.nan)
            h_pv = r.get(f'{tag}_hist_pv', np.nan)
            diff = r.get(f'{tag}_pv_diff', np.nan)
            diff_pct = r.get(f'{tag}_pv_diff_pct', np.nan)
            short = tag.replace('03', '').replace('_', '')
            print(f"      {short:>8}: Test PV={t_pv:7.2f}  Hist PV={h_pv:7.2f}  Diff={diff:+6.2f} ({diff_pct:+5.1f}% of range)")
        print()
    
    # Generate plots
    fig_radar = plot_radar_comparison(records, comparison_tags)
    fig_bar = plot_bar_comparison(records, comparison_tags)
    fig_heatmap = plot_heatmap_comparison(records, comparison_tags)
    
    if show:
        if fig_radar:
            fig_radar.show()
        if fig_bar:
            fig_bar.show()
        if fig_heatmap:
            fig_heatmap.show()
    
    return records, fig_radar, fig_bar, fig_heatmap


# ==============================================================================
# SELECT AN EPISODE TO ANALYZE
# ==============================================================================
SELECTED_EPISODE = test_episode_ids[0]
print(f"Available test episodes: {sorted(test_episode_ids)}")
print(f"\nSelected episode: {SELECTED_EPISODE}")

Available test episodes: [521, 522, 527, 528, 529, 530, 531, 533, 534, 542, 546, 548, 556, 557, 558, 560, 565, 567, 573, 580, 583, 584, 585, 586, 587, 588, 590, 594, 595, 596, 598, 601, 607]

Selected episode: 558


In [23]:
# Run the dashboard for the selected episode
records, fig_r, fig_b, fig_h = generate_episode_dashboard(SELECTED_EPISODE)


  Episode 558: Plant State (PV) Comparison Dashboard
  19 action timestamps to compare

  Action 1 @ 21:02 | Actual: 1071 ↑ 2.0 | Rec: 1071 ↑ 4.0 | Sim: 0.282 | Matched Ep: 468
       LIC1071: Test PV=   8.95  Hist PV=  39.92  Diff=-30.97 (-432.1% of range)
       LIC1016: Test PV=  54.65  Hist PV=  33.20  Diff=+21.46 (+433.0% of range)
       PIC1013: Test PV= 148.67  Hist PV= 162.84  Diff=-14.17 (-18.9% of range)

  Action 2 @ 21:40 | Actual: 1071 ↓ 2.0 | Rec: 1013 ↓ 2.0 | Sim: -0.050 | Matched Ep: 447
       LIC1071: Test PV=  60.91  Hist PV=  40.19  Diff=+20.72 (+289.2% of range)
       LIC1016: Test PV=  49.07  Hist PV=  42.51  Diff= +6.56 (+132.4% of range)
       PIC1013: Test PV= 169.61  Hist PV= 318.17  Diff=-148.56 (-197.7% of range)

  Action 3 @ 21:41 | Actual: 1071 ↓ 2.0 | Rec: 1071 ↑ 4.0 | Sim: 0.356 | Matched Ep: 468
       LIC1071: Test PV=  19.34  Hist PV=  39.92  Diff=-20.58 (-287.2% of range)
       LIC1016: Test PV=  56.37  Hist PV=  33.20  Diff=+23.17 (+467.6% of 

---
## Aggregate Analysis Across All Test Episodes

Build PV comparison data for ALL test episodes and produce:
- **Distribution of PV differences** per tag (box plots – as % of operating range)
- **Similarity score vs PV difference** (scatter – does the similarity metric actually capture PV similarity?)
- **Raw PV difference box plots** (in engineering units)

In [24]:
from tqdm import tqdm

# Build comparison data for ALL test episodes
print("Building comparison data for all test episodes...")
all_comparison_records = []

for ep_id in tqdm(test_episode_ids, desc="Processing episodes"):
    ep_records = build_episode_comparison_data(ep_id)
    all_comparison_records.extend(ep_records)

all_comp_df = pd.DataFrame(all_comparison_records)
print(f"\nTotal comparison records across all episodes: {len(all_comp_df)}")
print(f"Episodes with data: {all_comp_df['episode_id'].nunique()}")
print(f"\nCompiled columns: {len(all_comp_df.columns)}")

Building comparison data for all test episodes...


Processing episodes: 100%|██████████| 33/33 [00:00<00:00, 150.90it/s]


Total comparison records across all episodes: 210
Episodes with data: 33

Compiled columns: 39


In [25]:
# --- Plot 4: Box plot of PV differences (% of operating range) per tag ---

diff_pct_cols = [f'{tag}_pv_diff_pct' for tag in COMPARISON_TAGS]
tag_labels = [t.replace('03', '').replace('_', '') for t in COMPARISON_TAGS]

fig_box = go.Figure()
for col, label in zip(diff_pct_cols, tag_labels):
    vals = all_comp_df[col].dropna()
    fig_box.add_trace(go.Box(
        y=vals, name=label,
        boxmean='sd',
        hovertemplate=f'{label}<br>PV Diff: %{{y:.1f}}% of range<extra></extra>'
    ))

fig_box.add_hline(y=0, line_dash='dash', line_color='gray', opacity=0.5)
fig_box.update_layout(
    title='Distribution of PV Differences (% of Operating Range) Across All Test Episodes<br>'
          '<sub>0% = identical PV values | Wider box = more variation in plant state matching</sub>',
    yaxis_title='PV Difference (% of Operating Range)',
    height=500,
    showlegend=False,
)
fig_box.show()

# Print summary statistics
print("\n=== Summary Statistics: |PV Difference| per Tag ===")
print(f"{'Tag':<15} {'Mean |Diff%|':>12} {'Median |Diff%|':>14} {'Std':>10} {'% within ±5%':>14} {'% within ±10%':>14}")
print("-" * 85)
for col, label in zip(diff_pct_cols, tag_labels):
    vals = all_comp_df[col].dropna()
    abs_vals = vals.abs()
    within_5 = (abs_vals < 5).mean() * 100
    within_10 = (abs_vals < 10).mean() * 100
    print(f"{label:<15} {abs_vals.mean():>12.1f}% {abs_vals.median():>14.1f}% {abs_vals.std():>10.1f}% {within_5:>13.1f}% {within_10:>13.1f}%")


=== Summary Statistics: |PV Difference| per Tag ===
Tag             Mean |Diff%| Median |Diff%|        Std   % within ±5%  % within ±10%
-------------------------------------------------------------------------------------
LIC1071                230.2%          139.1%      232.7%           2.9%           6.2%
LIC1016                160.9%          137.9%      138.1%           2.9%           5.7%
PIC1013                 87.0%           46.8%      106.6%          28.1%          33.3%


In [26]:
# --- Plot 5: Similarity Score vs PV Difference (scatter) ---
# Does higher similarity score actually mean more similar PV values?

all_comp_df['mean_abs_pv_diff_pct'] = all_comp_df[
    [f'{tag}_pv_diff_pct' for tag in COMPARISON_TAGS]
].abs().mean(axis=1)

# Color by whether tag matched
all_comp_df['tag_matched'] = all_comp_df['tag_match'].apply(
    lambda x: 'Tag Matched' if x == 1 else ('Tag Mismatch' if x == 0 else 'Unknown')
)

fig_scatter = go.Figure()

for match_type, color in [('Tag Matched', 'green'), ('Tag Mismatch', 'red'), ('Unknown', 'gray')]:
    subset = all_comp_df[all_comp_df['tag_matched'] == match_type]
    if len(subset) == 0:
        continue
    fig_scatter.add_trace(go.Scatter(
        x=subset['similarity'],
        y=subset['mean_abs_pv_diff_pct'],
        mode='markers',
        name=match_type,
        marker=dict(color=color, size=8, opacity=0.6),
        hovertemplate=(
            'Episode: %{customdata[0]}<br>'
            'Similarity: %{x:.3f}<br>'
            'Mean |PV Diff|: %{y:.1f}% of range<br>'
            'Actual: %{customdata[1]}<br>'
            'Recommended: %{customdata[2]}'
            '<extra></extra>'
        ),
        customdata=subset[['episode_id', 'actual_source', 'rec_source']].values
    ))

fig_scatter.update_layout(
    title='Similarity Score vs Actual PV Difference<br>'
          '<sub>Lower y-axis = more similar PV values | Higher x-axis = higher similarity score</sub>',
    xaxis_title='Similarity Score (from approach)',
    yaxis_title='Mean |PV Difference| (% of Operating Range)',
    height=500,
    width=800,
)
fig_scatter.show()

corr = all_comp_df[['similarity', 'mean_abs_pv_diff_pct']].dropna().corr().iloc[0, 1]
print(f"\nCorrelation between similarity score and mean |PV diff %|: {corr:.3f}")
print(f"  (Negative = higher similarity → smaller PV differences, which is expected)")


Correlation between similarity score and mean |PV diff %|: 0.069
  (Negative = higher similarity → smaller PV differences, which is expected)


In [27]:
# --- Plot 6: Box plot of RAW PV differences (engineering units) per tag ---

raw_diff_cols = [f'{tag}_pv_diff' for tag in COMPARISON_TAGS]

fig_box_raw = go.Figure()
for col, label in zip(raw_diff_cols, tag_labels):
    vals = all_comp_df[col].dropna()
    fig_box_raw.add_trace(go.Box(
        y=vals, name=label,
        boxmean='sd',
        hovertemplate=f'{label}<br>PV Diff: %{{y:.2f}}<extra></extra>'
    ))

fig_box_raw.add_hline(y=0, line_dash='dash', line_color='gray', opacity=0.5)
fig_box_raw.update_layout(
    title='Distribution of Raw PV Differences (Engineering Units) Across All Test Episodes<br>'
          '<sub>0 = identical PV reading | Shows actual magnitude of plant state difference</sub>',
    yaxis_title='PV Difference (Test - Historical)',
    height=500,
    showlegend=False,
)
fig_box_raw.show()

# Print summary
print("\n=== Summary Statistics: Raw |PV Difference| per Tag ===")
print(f"{'Tag':<15} {'Mean |Diff|':>12} {'Median |Diff|':>14} {'Std':>10} {'Op Range':>10}")
print("-" * 65)
for tag, label in zip(COMPARISON_TAGS, tag_labels):
    col = f'{tag}_pv_diff'
    vals = all_comp_df[col].dropna()
    abs_vals = vals.abs()
    op_range = op_limits[tag]['range'] if tag in op_limits else 0
    print(f"{label:<15} {abs_vals.mean():>12.2f} {abs_vals.median():>14.2f} {abs_vals.std():>10.2f} {op_range:>10.2f}")


=== Summary Statistics: Raw |PV Difference| per Tag ===
Tag              Mean |Diff|  Median |Diff|        Std   Op Range
-----------------------------------------------------------------
LIC1071                16.49           9.97      16.68       7.17
LIC1016                 7.97           6.83       6.84       4.96
PIC1013                65.40          35.20      80.12      75.16


---
## Explore Individual Episodes

Run the dashboard for another specific episode. Change `EXPLORE_EPISODE` below.

In [28]:
# Change this to explore different episodes
EXPLORE_EPISODE = test_episode_ids[1] if len(test_episode_ids) > 1 else test_episode_ids[0]

records_2, fig_r2, fig_b2, fig_h2 = generate_episode_dashboard(EXPLORE_EPISODE)


  Episode 573: Plant State (PV) Comparison Dashboard
  6 action timestamps to compare

  Action 1 @ 14:23 | Actual: 1013 ↓ 2.0 | Rec: 1013 ↓ 2.0 | Sim: 0.934 | Matched Ep: 574
       LIC1071: Test PV=  47.47  Hist PV=  47.47  Diff= +0.00 ( +0.0% of range)
       LIC1016: Test PV=  45.69  Hist PV=  45.69  Diff= +0.00 ( +0.0% of range)
       PIC1013: Test PV= 318.18  Hist PV= 318.18  Diff= +0.00 ( +0.0% of range)

  Action 2 @ 14:24 | Actual: 1013 ↑ 0.9 | Rec: 1013 ↑ 0.9 | Sim: 0.953 | Matched Ep: 574
       LIC1071: Test PV=  34.50  Hist PV=  47.47  Diff=-12.97 (-181.0% of range)
       LIC1016: Test PV=  45.00  Hist PV=  45.69  Diff= -0.69 (-13.9% of range)
       PIC1013: Test PV= 318.18  Hist PV= 318.18  Diff= +0.00 ( +0.0% of range)

  Action 3 @ 14:41 | Actual: 1013 ↓ 4.0 | Rec: 1013 ↓ 4.0 | Sim: 0.953 | Matched Ep: 574
       LIC1071: Test PV=  43.75  Hist PV=  47.47  Diff= -3.72 (-51.9% of range)
       LIC1016: Test PV=  43.64  Hist PV=  45.69  Diff= -2.04 (-41.2% of range)
  